In [ ]:
from setup import setup,scale_features
import pandas as pd
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


In [ ]:
features = setup()

scaled_features = scale_features(features)

featues_gmm = pd.DataFrame(scaled_features, index=features.index, columns=features.columns)

In [ ]:
#calc bayesian information criterion for GMM
bic = []
for k in range(1, 20):
    gmm = GaussianMixture(n_components=k, random_state=42)
    gmm.fit(scaled_features)
    bic.append(gmm.bic(scaled_features))

plt.plot(range(1, 20), bic, marker='o')
plt.title('BIC for GMM')
plt.xlabel('Number of clusters')
plt.ylabel('BIC')
plt.show()


In [ ]:
# run GMM clustering
gmm = GaussianMixture(n_components=4, random_state=42, covariance_type='full')
regime = gmm.fit_predict(scaled_features)

featues_gmm['regime'] = regime

# plot regimes
plt.figure(figsize=(12, 6))
plt.scatter(features.index, featues_gmm["avg_return"], c=regime, cmap='viridis', marker='o')
plt.title('GMM QQQ Price with Market Regimes')
plt.xlabel('Date')
plt.ylabel('Price')
plt.colorbar(label='Regime', ticks=range(4))
plt.show()

In [ ]:
pca = PCA(n_components=3)
features_pca = pca.fit_transform(scaled_features)
regime = gmm.fit_predict(features_pca)

scaled_features_pca = featues_gmm.copy()
scaled_features_pca["regime"] = regime

plt.figure(figsize=(12, 6))
plt.scatter(features_pca[:, 0], features_pca[:, 1], c=regime, cmap='viridis', marker='o')
plt.title('PCA of Market Regimes')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.colorbar(label='Regime')
plt.show()

    

In [ ]:
scaled_features_pca.assign(regime=regime).groupby('regime').mean()

In [ ]:
scaled_features_pca["future_return"] = scaled_features_pca["avg_return"].shift(-1)
scaled_features_pca.assign(regime=regime).groupby('regime')["future_return"].mean()